OFC electrodes that increase beta power during encoding of positive RPEs (all trials) also increase beta power for subsequent hits (half of trials), and increase beta power for hits (have RPEs) vs. false-alarms (no RPEs) following CUE presentation. This suggests a potentially important role for OFC beta oscillations in binding RPEs to enhanced memory. 

In this notebook, I want to do some more due diligence on OFC beta oscillations during encoding and cue.  

To that end, I want to:


1. plot some oscillations

2. characterize their PSD (using FOOOF) 

3. characterize their rise-peak asymmetry 

4. characterize bursts (length?) 

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 
from ast import literal_eval


# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest

from neurodsp.plts.spectral import plot_power_spectra


In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')
sys.path.append('/hpc/users/qasims01/resources/abcTau')


In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df.csv')
combined_df = pd.read_csv(f'{load_dir}/combined_df.csv')

# Filter just to the participants in this notebook
learn_df = learn_df[learn_df.participant.isin(subj_ids)]
combined_df = combined_df[combined_df.participant.isin(subj_ids)]


# I enter the day information in stupidly 
combined_df.condition = combined_df.condition.apply(lambda x: 'Day1' if x=='day_1' else 'Day2' if x=='day_2' else np.nan)
combined_df['recog_time'] = combined_df['recog_time'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)
combined_df['baseline_start_mem'] = combined_df['baseline_start_mem'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)

# add RPE to the combined df dataframe: 

combined_df['rpe'] = np.nan
for ix, row in learn_df.iterrows():
    combined_df.loc[(combined_df.img_path==row.img_path) & (combined_df.participant==row.participant), 'rpe'] = row.rpe

In [ ]:
%%time
# select event of interest
ev_dict = {'feedback_start': [0, 1.5]}

# select ROI
rois = ['OFC'] 

progress_bar = tqdm(subj_ids, ascii=True, desc='Computing PSD for all trials and subsetting by conditions')

psd_group_data = {f'{a}': [] for a in rois}

for subj_id in progress_bar:
    # Load one person's data 
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            if not os.path.exists(f'{filepath}/plots'):
                os.makedirs(f'{filepath}/plots')
            if not os.path.exists(f'{filepath}/dfs'):
                os.makedirs(f'{filepath}/dfs')

            mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)
            mne_data_reref.metadata = learn_df[learn_df.participant==subj_id]
            
            mne_data_reref.crop(tmin=ev_dict[event][0], tmax=ev_dict[event][-1])
            
            # NAN out the bad data
            # THE following will now LOAD in dataframes that indicate IED and artifact time points in your data
            IED_df = pd.read_csv(f'{load_path}/{event}_IED_df.csv') 
            artifact_df = pd.read_csv(f'{load_path}/{event}_artifact_df.csv') 

            # Now, let's iterate through each channel, and each ied/artifact, and NaN 100 ms before and after these timepoints
            for ch_ix, ch_name in enumerate(mne_data_reref.ch_names): 
                ied_ev_list = IED_df[ch_name].dropna().index.tolist()
                artifact_ev_list = artifact_df[ch_name].dropna().index.tolist() 
                for ev_ in ied_ev_list: 
                    for ied_ in literal_eval(IED_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((ied_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((ied_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan
                for ev_ in artifact_ev_list: 
                    for artifact_ in literal_eval(artifact_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((artifact_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((artifact_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan


            mne_data_reref._data = mne_data_reref._data * 1000000
            save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            # Get electrode df 
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
                    
            progress_bar = tqdm(elec_df.label, ascii=True, desc='Computing PSDs')

            for elec_ix, chan in enumerate(progress_bar):
                region = elec_df.salman_region.iloc[elec_ix]
                if region in rois:
                    # compute PSD here 
                    epo_spectrum = mne_data_reref.copy().pick([chan]).compute_psd(method='multitaper',
                                                fmin=2, 
                                                fmax=200,
                                                tmin=ev_dict[event][0],
                                                tmax=ev_dict[event][-1],
                                                verbose=False)
                                                
#                     psds, freqs = epo_spectrum.get_data(return_freqs=True)
                    psds = epo_spectrum._data
                    freqs = epo_spectrum.freqs
                    # average across trials and squeeze out the empty axis 
                    avg_psd = np.nanmean(psds, axis=0)[0,:]
                    avg_psd = avg_psd/np.nanmean(avg_psd)
#                     avg_psd = (avg_psd - np.nanmin(avg_psd)) / (np.nanmax(avg_psd) - np.nanmin(avg_psd)) 
                    psd_group_data[region].append(avg_psd)                    
                    

In [ ]:
for psd in psd_group_data['OFC']:
    plt.plot(freqs[freqs<40], np.log10(psd[np.where(freqs<40)]), 'k', alpha = 0.1)
plt.plot(freqs[freqs<40], np.log10(np.nanmean(psd_group_data['OFC'], axis=0)[np.where(freqs<40)]), 'r')


In [ ]:
%%time
ev_dict = {'feedback_start': [0, 1.5]}

# Define peak_width, peak_height, peak_threshold, max_n_peaks
FOOOF_kwargs = {'peak_width_limits': [1, 8], 
               'min_peak_height': 0.05,
                'peak_threshold': 1.,
                'max_n_peaks': 6,
               'freq_range': [2, 40]}
 
# Define our frequency bands of interest
band_dict = {'theta' : [2, 8],
               'alpha' : [8, 13],
               'beta' : [13, 30]}

conditions = ['rpe>0',
             'rpe<1']

progress_bar = tqdm(subj_ids, ascii=True, desc='Computing FOOOF for all trials and subsetting by conditions')

for subj_id in progress_bar:
    # Load one person's data 
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            if not os.path.exists(f'{filepath}/plots'):
                os.makedirs(f'{filepath}/plots')
            if not os.path.exists(f'{filepath}/dfs'):
                os.makedirs(f'{filepath}/dfs')

            mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)
            mne_data_reref.metadata = learn_df[learn_df.participant==subj_id]
            
            mne_data_reref.crop(tmin=ev_dict[event][0], tmax=ev_dict[event][-1])
            
            # NAN out the bad data
            # THE following will now LOAD in dataframes that indicate IED and artifact time points in your data
            IED_df = pd.read_csv(f'{load_path}/{event}_IED_df.csv') 
            artifact_df = pd.read_csv(f'{load_path}/{event}_artifact_df.csv') 

            # Now, let's iterate through each channel, and each ied/artifact, and NaN 100 ms before and after these timepoints
            for ch_ix, ch_name in enumerate(mne_data_reref.ch_names): 
                ied_ev_list = IED_df[ch_name].dropna().index.tolist()
                artifact_ev_list = artifact_df[ch_name].dropna().index.tolist() 
                for ev_ in ied_ev_list: 
                    for ied_ in literal_eval(IED_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((ied_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((ied_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan
                for ev_ in artifact_ev_list: 
                    for artifact_ in literal_eval(artifact_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((artifact_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((artifact_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan
                        

            save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            # Get electrode df 
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
            
            # # Run the FOOOF code in parallel
            Parallel(n_jobs=-1, verbose=1)(delayed(analysis_utils.compute_FOOOF_parallel)(chan, mne_data_reref, 
                                                                                         subj_id, 
                                                                                         elec_df, 
                                                                                         event, 
                                                                                         ev_dict, 
                                                                                         band_dict, 
                                                                                         conditions, 
                                                                                         do_plot=True, 
                                                                                         save_path='/sc/arion/projects/guLab/Salman/EphysAnalyses',
                                                                                         do_save=True, 
                                                                                         **FOOOF_kwargs) for chan in mne_data_reref.ch_names)


In [ ]:
# Load the dataframes: 
chan_dfs = [] 
for subj_id in subj_ids[0:1]:
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
            for chan in tqdm(elec_df.label.unique(), ascii=True, desc='Loading FOOOF dataframes'): 
                chan_df = pd.read_csv(f'{filepath}/dfs/{chan}_df.csv')
                chan_dfs.append(chan_df)
chan_dfs = pd.concat(chan_dfs)



In [ ]:
chan_dfs['fband'] = chan_dfs.apply(lambda x: 'theta' if ((x.frequency>=2) & (x.frequency<8)) else 'alpha' if ((x.frequency>=8) & (x.frequency<13)) else 'beta' if ((x.frequency>=13) & (x.frequency<=30)) else np.nan, axis=1)


In [ ]:
# roi_df = chan_dfs[chan_dfs['region'].isin(['HPC', 'AMY', 'OFC', 'dmPFC', 'ACC'])]
# roi_df = roi_df[(roi_df.cond=='all')]
# roi_df['beta_peak'] = 0

In [ ]:
# # these are the electrodes with >= 1 peak in theta frequencies
# beta_channel_ids = roi_df[((roi_df['peak_freq'] > 13 ) & (roi_df['peak_freq'] <= 30))][['participant', 'channel']].drop_duplicates()

# for row in beta_channel_ids.itertuples():
#     roi_df.loc[((roi_df.participant==row.participant) & (roi_df.channel==row.channel)), 'beta_peak'] = 1
    

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), dpi=300, sharey=True)

sns.lineplot(data=roi_df[roi_df.region=='ACC'], 
             x='frequency', 
             y='PSD_corrected', 
             errorbar='se',  
             ax=ax[0])
sns.lineplot(data=roi_df[roi_df.region=='dmPFC'], 
             x='frequency', 
             y='PSD_corrected', 
             errorbar='se',  
             ax=ax[1])
sns.lineplot(data=roi_df[roi_df.region=='OFC'], 
             x='frequency', 
             y='PSD_corrected', 
             errorbar='se', 
             ax=ax[2])

ax[0].hlines(0, roi_df.frequency.min(), roi_df.frequency.max())
ax[1].hlines(0, roi_df.frequency.min(), roi_df.frequency.max())
ax[2].hlines(0, roi_df.frequency.min(), roi_df.frequency.max())
            
fig.tight_layout()
sns.despine()


Of all frontal cortex regions, OFC has the most profound beta oscillations during the encoding period

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4), dpi=300, sharey=True)

sns.lineplot(data=roi_df[roi_df.region=='HPC'], 
             x='frequency', 
             y='PSD_corrected', 
             errorbar='se',  
             ax=ax[0])
sns.lineplot(data=roi_df[roi_df.region=='AMY'], 
             x='frequency', 
             y='PSD_corrected', 
             errorbar='se',  
             ax=ax[1])


ax[0].hlines(0, roi_df.frequency.min(), roi_df.frequency.max())
ax[1].hlines(0, roi_df.frequency.min(), roi_df.frequency.max())
            
fig.tight_layout()
sns.despine()


Now, let's identify bursts of beta oscillations and plot those data: 

1. In the trial-wise TFR data for MS036 or MS026, grab the beta range (13-30)

2. Identify time points where z>0.75*max(z) for at least 231 ms

3. Plot the raw data from the epochs that have those bursts



In [ ]:
progress_bar = tqdm(subj_ids, ascii=True, desc='Saving TFR dataframes')
analysis_evs = ['feedback_start'] 
rois = ['OFC']
freqs = np.logspace(*np.log10([2, 200]), num=30)


for subj_id in ['MS036']:
    for day in ['Day1']:
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for event in analysis_evs:

            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            if not os.path.exists(f'{filepath}/{event}/plots'):
                os.makedirs(f'{filepath}/{event}/plots')
            if not os.path.exists(f'{filepath}/{event}/dfs'):
                os.makedirs(f'{filepath}/{event}/dfs')

            power_epochs = mne.time_frequency.read_tfrs(f'{filepath}/{event}-tfr.h5')[0]
            power_epochs.metadata = learn_df[(learn_df.participant==subj_id)]
            
            save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            # Get electrode df 
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
                  
            progress_bar = tqdm(elec_df.label, ascii=True, desc='Computing PSDs')
            
            load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
            mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)
            mne_data_reref.metadata = learn_df[learn_df.participant==subj_id]
            
            mne_data_reref.crop(tmin=ev_dict[event][0], tmax=ev_dict[event][-1])

            for elec_ix, chan in enumerate(progress_bar):
                region = elec_df.salman_region.iloc[elec_ix]
                if region in rois:
                    
                    beta_pow = np.nanmean(np.squeeze(power_epochs.copy()._data[:, elec_ix, np.where((freqs>=13) & (freqs<30))[0], :]), 1)
                    
            

                    

In [ ]:
elec_df[elec_df.salman_region=='OFC']

In [ ]:
elec_ix = 60

In [ ]:
beta_pow = np.nanmean(np.squeeze(power_epochs.copy()._data[:, elec_ix, np.where((freqs>=13) & (freqs<30))[0], :]), 1)



In [ ]:
def find_clusters(vector, threshold, min_length):
    clusters = []
    current_cluster = []
    
    for data in vector:
        if data > threshold:
            current_cluster.append(data)
        else:
            if len(current_cluster) >= min_length:
                clusters.append(current_cluster)
            current_cluster = []
    
    if len(current_cluster) >= min_length:
        clusters.append(current_cluster)
    
    return clusters

In [ ]:
for ix, trial in enumerate(beta_pow):
    print(ix)
    print(find_clusters(trial,0.5, 200))

In [ ]:
# plot_data = mne_data_reref.copy().filter(13, 30)

In [ ]:
tfr_data = power_epochs._data[61, elec_ix, :, :]

fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)
times = np.linspace(0, 1.5, tfr_data.shape[-1])

im = ax.imshow(tfr_data,
              aspect='auto', origin='lower', cmap='Spectral_r', interpolation = 'bicubic', vmax=3)            
fig.colorbar(im, ax=ax)
freq_res = 5
time_res = 250 
ax.set(yticks=np.arange(0, len(freqs))[::freq_res], 
       yticklabels=freqs[::freq_res].astype(int), 
       xticks = np.arange(len(times))[::time_res],
       xticklabels= power_epochs.times[::time_res])


MS036
elec 28, trial 60

elec 60, trial 61

In [ ]:
trial = 61

fig, ax = plt.subplots(2, 1, figsize=(15, 4), dpi=300, sharex=True)
ax[0].plot(power_epochs.times, beta_pow[trial, :])
ax[1].plot(power_epochs.times, mne_data_reref._data[trial, elec_ix, :] * 1000000, 'k')
ax[1].plot(power_epochs.times, plot_data._data[trial, elec_ix, :] * 1000000, 'gray')

sns.despine()
